## Silver Data Quality Checks — Step 1

In [0]:
# Customer null check
print("Customers with NULL customer_id:",
      spark.table("silver_customers")
           .filter("customer_id IS NULL")
           .count())

# Account null check
print("Accounts with NULL account_id:",
      spark.table("silver_accounts")
           .filter("account_id IS NULL")
           .count())

# Transaction null check
print("Transactions with NULL transaction_id:",
      spark.table("silver_transactions")
           .filter("transaction_id IS NULL")
           .count())

# Branch null check
print("Branches with NULL branch_id:",
      spark.table("silver_branches")
           .filter("branch_id IS NULL")
           .count())

Customers with NULL customer_id: 0
Accounts with NULL account_id: 0
Transactions with NULL transaction_id: 0
Branches with NULL branch_id: 0


####Step 2 — Check duplicates


In [0]:
from pyspark.sql.functions import count

print("Duplicate customers:",
      spark.table("silver_customers")
           .groupBy("customer_id")
           .count()
           .filter("count > 1")
           .count())

print("Duplicate accounts:",
      spark.table("silver_accounts")
           .groupBy("account_id")
           .count()
           .filter("count > 1")
           .count())

print("Duplicate transactions:",
      spark.table("silver_transactions")
           .groupBy("transaction_id")
           .count()
           .filter("count > 1")
           .count())

print("Duplicate branches:",
      spark.table("silver_branches")
           .groupBy("branch_id")
           .count()
           .filter("count > 1")
           .count())

Duplicate customers: 0
Duplicate accounts: 0
Duplicate transactions: 0
Duplicate branches: 0


#### Step 3 — Check invalid transaction amounts

In [0]:
from pyspark.sql.functions import *

df_customers = spark.table("silver_customers")
df_accounts = spark.table("silver_accounts")
df_transactions = spark.table("silver_transactions")

invalid_amounts = df_transactions.filter(
    (col("amount").isNull()) | (col("amount") <= 0)
)

print("Invalid transaction amounts:", invalid_amounts.count())

display(invalid_amounts)

Invalid transaction amounts: 0


transaction_id,account_id,transaction_date,transaction_type,amount,channel


#### Step 4 — Check orphan transactions

In [0]:
df_accounts = spark.table("silver_accounts")
df_transactions = spark.table("silver_transactions")

orphan_transactions = (
    df_transactions
    .join(
        df_accounts.select("account_id"),
        on="account_id",
        how="left_anti"
    )
)

print("Orphan transactions:", orphan_transactions.count())

display(orphan_transactions)

Orphan transactions: 0


account_id,transaction_id,transaction_date,transaction_type,amount,channel


#### Step 5 — Check orphan accounts

In [0]:
df_customers = spark.table("silver_customers")
df_accounts = spark.table("silver_accounts")

orphan_accounts = (
    df_accounts
    .join(
        df_customers.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

print("Orphan accounts:", orphan_accounts.count())

display(orphan_accounts)

Orphan accounts: 0


customer_id,account_id,account_type,opening_date,balance
